Parameter locking and error handling

In [ ]:
import os
os.environ["GEMINI_API_KEY"] = "CPSP_AD"

In [1]:
import json

findings = {
    "cleaned_total_revenue_inr": 97358.3,
    "raw_total_revenue_inr": 99860.2,
    "duplicate_reconciliation_delta_inr": 2501.9,
    "return_rate_by_payment": {
        "COD": 44.4,
        "CARD": 14.7,
        "UPI": 18.9
    },
    "highest_risk_segment": {
        "payment_method": "COD",
        "city_tier": 2,
        "return_rate_pct": 54.5
    },
    "true_peak_month": {
        "month": "2026-03",
        "revenue_inr": 20318.9
    },
    "outlier_inflated_month": {
        "month": "2026-01",
        "apparent_revenue_inr": 29582.1,
        "corrected_revenue_inr": 11637.1
    }
}

with open("findings.json", "w", encoding="utf-8") as f:
    json.dump(findings, f, indent=2, ensure_ascii=False)

print("findings.json created successfully!")

findings.json created successfully!


In [ ]:
import os
from google import genai


def generate_scr_narrative(findings: dict) -> dict:

    system_instruction = """
You are a senior data analyst writing for Mamaearth's regional ops and finance heads.

Write a concise business narrative with exactly three labelled sections:

Situation
Complication
Resolution

Rules:
- Every number must come from the supplied findings.
- Use exactly the same values provided.
- Do not invent statistics, percentages, revenue figures, months, or trends.
- Base all conclusions only on the supplied findings.
"""

    user_prompt = f"""
Use the following verified findings.

Cleaned Revenue: ₹{findings["cleaned_total_revenue_inr"]}
Raw Revenue: ₹{findings["raw_total_revenue_inr"]}
Duplicate Reconciliation Delta: ₹{findings["duplicate_reconciliation_delta_inr"]}

Return Rate by Payment:
{findings["return_rate_by_payment"]}

Highest Risk Segment:
{findings["highest_risk_segment"]}

True Peak Month:
{findings["true_peak_month"]}

Outlier Inflated Month:
{findings["outlier_inflated_month"]}

Create an SCR narrative.
"""

    api_key = os.getenv("CPSP_AD")

    if not api_key:
        return {
            "status": "error",
            "narrative": None,
            "message": "GEMINI_API_KEY not configured"
        }

    try:

        client = genai.Client(
            api_key=api_key
        )

        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=user_prompt,
            config={
                # temperature=0.0 because this is a factual
                # business report, not creative writing.
                "temperature": 0.0,
                "max_output_tokens": 300,
                "system_instruction": system_instruction,

                # 15-second timeout = 15000 milliseconds.
                "http_options": {
                    "timeout": 15000
                }
            }
        )

        return {
            "status": "success",
            "narrative": response.text,
            "tokens": 0
        }

    except Exception as err:

        return {
            "status": "error",
            "narrative": None,
            "message": str(err)
        }
result = generate_scr_narrative(findings)
print(result)

{'status': 'error', 'narrative': None, 'message': 'GEMINI_API_KEY not configured'}


Offline fallback path

In [ ]:
def generate_scr_narrative_offline(findings: dict) -> dict:
    """
    Fully deterministic offline fallback.
    Uses only the supplied findings and makes no network/API call.
    """

    narrative = f"""Situation

Cleaned revenue is ₹{findings["cleaned_total_revenue_inr"]}, compared with raw revenue of ₹{findings["raw_total_revenue_inr"]}. Return rates are COD at {findings["return_rate_by_payment"]["COD"]}%, CARD at {findings["return_rate_by_payment"]["CARD"]}%, and UPI at {findings["return_rate_by_payment"]["UPI"]}%.

Complication

The duplicate reconciliation delta is ₹{findings["duplicate_reconciliation_delta_inr"]}. The highest-risk segment is {findings["highest_risk_segment"]["payment_method"]} in city tier {findings["highest_risk_segment"]["city_tier"]}, with a return rate of {findings["highest_risk_segment"]["return_rate_pct"]}%. The apparent revenue for {findings["outlier_inflated_month"]["month"]} was ₹{findings["outlier_inflated_month"]["apparent_revenue_inr"]}, while corrected revenue was ₹{findings["outlier_inflated_month"]["corrected_revenue_inr"]}.

Resolution

Use the cleaned revenue of ₹{findings["cleaned_total_revenue_inr"]} for reporting. The true peak month was {findings["true_peak_month"]["month"]}, with revenue of ₹{findings["true_peak_month"]["revenue_inr"]}. Review the highest-risk segment as part of operational follow-up."""

    return {
        "status": "success",
        "narrative": narrative,
        "tokens": 0
    }

In [ ]:
offline_result = generate_scr_narrative(findings)

print(offline_result)

{'status': 'error', 'narrative': None, 'message': 'GEMINI_API_KEY not configured'}
